In [1]:
!pip install pyspark

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *

In [3]:
spark = SparkSession.builder \
    .appName("Week6Assignment") \
    .getOrCreate()

print("Spark Started Successfully!")

Spark Started Successfully!


In [4]:
#creating a sample dataset

import pandas as pd

data = {
    "product_id":[101,102,103,104,105],
    "category":["Electronics","Furniture","Electronics","Clothing","Electronics"],
    "price":[500,1200,750,400,950],
    "status":["Completed","Pending","Completed","Completed","Pending"],
    "amount":[1500,800,2000,1100,500],
    "region":["North","South","East","North","West"],
    "priority":["High","Low","Medium","High","Low"],
    "base_price":[500,1200,750,400,950]
}

df = pd.DataFrame(data)

df.to_csv("source.csv", index=False)

print(df)

   product_id     category  price     status  amount region priority  \
0         101  Electronics    500  Completed    1500  North     High   
1         102    Furniture   1200    Pending     800  South      Low   
2         103  Electronics    750  Completed    2000   East   Medium   
3         104     Clothing    400  Completed    1100  North     High   
4         105  Electronics    950    Pending     500   West      Low   

   base_price  
0         500  
1        1200  
2         750  
3         400  
4         950  


In [5]:
df = spark.read.csv(
    "source.csv",
    header=True,
    inferSchema=True
)

df.show()

+----------+-----------+-----+---------+------+------+--------+----------+
|product_id|   category|price|   status|amount|region|priority|base_price|
+----------+-----------+-----+---------+------+------+--------+----------+
|       101|Electronics|  500|Completed|  1500| North|    High|       500|
|       102|  Furniture| 1200|  Pending|   800| South|     Low|      1200|
|       103|Electronics|  750|Completed|  2000|  East|  Medium|       750|
|       104|   Clothing|  400|Completed|  1100| North|    High|       400|
|       105|Electronics|  950|  Pending|   500|  West|     Low|       950|
+----------+-----------+-----+---------+------+------+--------+----------+



In [6]:
df.printSchema()

root
 |-- product_id: integer (nullable = true)
 |-- category: string (nullable = true)
 |-- price: integer (nullable = true)
 |-- status: string (nullable = true)
 |-- amount: integer (nullable = true)
 |-- region: string (nullable = true)
 |-- priority: string (nullable = true)
 |-- base_price: integer (nullable = true)



In [7]:
df.filter(df.category=="Electronics") \
.select("product_id","price") \
.show()

+----------+-----+
|product_id|price|
+----------+-----+
|       101|  500|
|       103|  750|
|       105|  950|
+----------+-----+



In [8]:
df.filter(df.category=="Electronics") \
.select("product_id","price") \
.show()

+----------+-----+
|product_id|price|
+----------+-----+
|       101|  500|
|       103|  750|
|       105|  950|
+----------+-----+



In [9]:
from pyspark.sql.functions import col

df = df.withColumnRenamed(
    "category",
    "old_name"
)

df = df.withColumnRenamed(
    "old_name",
    "new_name"
)

df = df.withColumn(
    "price",
    col("price").cast("double")
)

df.show()

+----------+-----------+------+---------+------+------+--------+----------+
|product_id|   new_name| price|   status|amount|region|priority|base_price|
+----------+-----------+------+---------+------+------+--------+----------+
|       101|Electronics| 500.0|Completed|  1500| North|    High|       500|
|       102|  Furniture|1200.0|  Pending|   800| South|     Low|      1200|
|       103|Electronics| 750.0|Completed|  2000|  East|  Medium|       750|
|       104|   Clothing| 400.0|Completed|  1100| North|    High|       400|
|       105|Electronics| 950.0|  Pending|   500|  West|     Low|       950|
+----------+-----------+------+---------+------+------+--------+----------+



In [10]:
df.filter(
    (df.status=="Completed") &
    (df.amount>1000)
).show()

+----------+-----------+-----+---------+------+------+--------+----------+
|product_id|   new_name|price|   status|amount|region|priority|base_price|
+----------+-----------+-----+---------+------+------+--------+----------+
|       101|Electronics|500.0|Completed|  1500| North|    High|       500|
|       103|Electronics|750.0|Completed|  2000|  East|  Medium|       750|
|       104|   Clothing|400.0|Completed|  1100| North|    High|       400|
+----------+-----------+-----+---------+------+------+--------+----------+



In [11]:
df = df.withColumn(
    "final_price",
    col("base_price")*1.18
)

df.show()

+----------+-----------+------+---------+------+------+--------+----------+-----------+
|product_id|   new_name| price|   status|amount|region|priority|base_price|final_price|
+----------+-----------+------+---------+------+------+--------+----------+-----------+
|       101|Electronics| 500.0|Completed|  1500| North|    High|       500|      590.0|
|       102|  Furniture|1200.0|  Pending|   800| South|     Low|      1200|     1416.0|
|       103|Electronics| 750.0|Completed|  2000|  East|  Medium|       750|      885.0|
|       104|   Clothing| 400.0|Completed|  1100| North|    High|       400|      472.0|
|       105|Electronics| 950.0|  Pending|   500|  West|     Low|       950|     1121.0|
+----------+-----------+------+---------+------+------+--------+----------+-----------+



In [12]:
df.filter(
    df.product_id.isNotNull()
).write.mode("overwrite") \
.csv("output_csv", header=True)

In [13]:
df.write.mode("overwrite") \
.parquet("output_parquet")